# Patent Disruption Trend — year-by-year disruption (CD) and citer profile (F/E/G) per patent

For every US utility patent with at least one citer, its **disruption index and citer profile as
they evolve**, anchored at the **grant year**: one row per patent × year in which anything changed,
carrying

- `ni_new, nj_new, nk_new` — citing / co-citing patents that appeared **in that year**
  ($n_i$: cites the patent but none of its references; $n_j$: cites the patent **and** ≥ 1 of its
  references; $n_k$: cites ≥ 1 of its references but not the patent);
- `ni, nj, nk` — the **cumulative** counts from the grant year through that year;
- `CD` — $\dfrac{n_i-n_j}{n_i+n_j+n_k}$ on the cumulative counts, i.e. the disruption index with a
  citation window of exactly `yrs_since_grant` years;
- `F, E, G` — the Foundational / Extensional / Generalizational shares of the citers that have
  arrived by that year (`up` = references shared with the patent, `down` = citers of the patent the
  citer also cites, both counted within the window), NaN until the first citer.

This is the time axis that `patent_disruption.ipynb` collapses into four windows: the row at
`yrs_since_grant = w` (or the last row before it) **is** `CD_w` / `F_w` / `E_w` / `G_w`, and the
last row is the `_all` value. Section 4 checks that identity on **every** focal patent against
`patent_disruption.parquet`. Same relationship as `patent_citation_trend` has to `patent_citation`.
(Not to be confused with `patent_feg_disruption_trend`, which averages the per-window values by
grant year; this file is per patent.)

## Raw / input data
```
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_patent.tsv.zip              # patent_id, patent_type, patent_date -> grant year, utility filter
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_us_patent_citation.tsv.zip  # patent_id (citing), citation_patent_id (cited)
/project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption.parquet      # per-window CD/F/E/G/ni/nj/nk; the check target
```
Only **utility↔utility** citations are kept; directed `(citing, cited)` pairs are de-duplicated and
self-loops dropped — the same graph `patent_disruption.ipynb` builds, so the two tables agree exactly.

## Engine
The `paper_disruption` numba kernel, re-cut along time: each focal patent's citers are classified
i / j once (a citer's type does not depend on the window) and binned by citing grant year; the
distinct citers of its references are binned by their grant year; prefix sums over years give the
cumulative $n_i, n_j, n_k$ and CD. F/E/G do depend on the window (`down` counts co-cited citers
inside it), so each citer's `down` is re-counted at every age it is present.

## Output
`/project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption_trend.parquet` —
`patent_id, grant_year, cite_year, yrs_since_grant, ni_new, nj_new, nk_new, ni, nj, nk, CD, F, E, G`
(one row per patent × year with ≥ 1 new citer or co-citing patent; `CD, F, E, G` are cumulative to that year).

`/project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption_trend_summary.parquet` —
`grant_year, yrs_since_grant, n_<m>, <m>_mean` for `m` in CD/F/E/G: mean cumulative outcomes by cohort and age.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import numba
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
DATA_DIR = pv.GRANTED
OUT_FP  = pv.out('patent_disruption_trend.parquet')
SUM_FP  = pv.out('patent_disruption_trend_summary.parquet')
DISR_FP = pv.out('patent_disruption.parquet')    # per-window table: the check target in §4
MIN_YEAR, MAX_YEAR = 1976, 2025                   # as in patent_disruption
BATCH   = 1_000_000                               # focal patents per kernel call / write
NT = int(os.environ.get('SLURM_CPUS_PER_TASK', numba.config.NUMBA_NUM_THREADS))
numba.set_num_threads(min(NT, numba.config.NUMBA_NUM_THREADS))
print('numba threads:', numba.get_num_threads())
pv.preflight('patent_disruption')                 # same inputs as patent_disruption
print('output ->', OUT_FP)

numba threads: 16
granted    : /project/jevans/Dawoon/Science of Science/PatentView/Granted   (35 zip files)
pregranted : /project/jevans/Dawoon/Science of Science/PatentView/Pregranted   (25 files)
output     : /project/jevans/Dawoon/Science of Science/PatentView/output

  patent_disruption               OK
output -> /project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption_trend.parquet


## 1. Load raw grant years (utility patents)

In [2]:
%%time
dp = pd.read_csv(os.path.join(DATA_DIR, 'g_patent.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'patent_type', 'patent_date'],
                 dtype={'patent_id': str, 'patent_type': str})
dp['grant_year'] = pd.to_datetime(dp['patent_date'], errors='coerce').dt.year
dp = dp[dp['patent_type'] == 'utility'].dropna(subset=['grant_year'])
dp['grant_year'] = dp['grant_year'].astype(int)
dp = dp[dp['grant_year'].between(MIN_YEAR, MAX_YEAR)]
pid2year = dict(zip(dp['patent_id'], dp['grant_year']))
util = set(pid2year)
print(f'utility patents with grant year: {len(pid2year):,}')
del dp; gc.collect()

utility patents with grant year: 8,531,961


## 2. Load raw citations → utility↔utility edges → integer codes → int32 CSR

Identical to `patent_disruption.ipynb` §2 (de-duplicated directed pairs, self-loops dropped), so the trend is computed on exactly the graph whose per-window values it must reproduce.

In [3]:
%%time
cf, ct, seen, t0 = [], [], 0, time.time()
for ch in pd.read_csv(os.path.join(DATA_DIR, 'g_us_patent_citation.tsv.zip'), sep='\t',
                      usecols=['patent_id', 'citation_patent_id'], dtype=str, chunksize=5_000_000):
    ch = ch.dropna(subset=['citation_patent_id'])
    m = ch['patent_id'].isin(util) & ch['citation_patent_id'].isin(util)
    if m.any():
        cf.append(ch['patent_id'].values[m.values]); ct.append(ch['citation_patent_id'].values[m.values])
    seen += len(ch)
citing_pid = np.concatenate(cf); cited_pid = np.concatenate(ct); del cf, ct; gc.collect()
print(f'[{time.time()-t0:.0f}s] scanned {seen:,} rows -> {len(citing_pid):,} utility<->utility edges')

allids = pd.Index(pd.unique(np.concatenate([citing_pid, cited_pid])))
codemap = {p: i for i, p in enumerate(allids)}
c_from = pd.Series(citing_pid).map(codemap).to_numpy(np.int64)
c_to   = pd.Series(cited_pid).map(codemap).to_numpy(np.int64)
del citing_pid, cited_pid, codemap; gc.collect()
uni = np.asarray(allids); n = len(uni)

# De-duplicate directed (citing, cited) pairs, then drop self-loops -- as patent_disruption does.
n_raw = len(c_from)
key = c_from * np.int64(n) + c_to
del c_from, c_to; gc.collect()
key = np.unique(key); n_uniq = len(key)
c_from = (key // n).astype(np.int64); c_to = (key % n).astype(np.int64); del key; gc.collect()
loop = c_from != c_to; n_self = int((~loop).sum())
c_from = c_from[loop]; c_to = c_to[loop]; del loop; gc.collect()
year = np.array([pid2year[p] for p in uni], dtype=np.int32)
print(f'{n:,} patents in citation graph; {len(c_from):,} unique directed edges '
      f'(from {n_raw:,} raw rows: {n_raw - n_uniq:,} duplicates, {n_self:,} self-loops removed)')

def build_csr(src, dst, n):
    order = np.argsort(src, kind='stable'); s = src[order]; d = dst[order].astype(np.int32)
    indptr = np.zeros(n + 1, np.int64); np.add.at(indptr, s + 1, 1); np.cumsum(indptr, out=indptr)
    return indptr, d

out_ptr, out_idx = build_csr(c_from, c_to, n)      # refs(x)
in_ptr,  in_idx  = build_csr(c_to,  c_from, n)     # citers(x)
del c_from, c_to; gc.collect()
YMAX = int(year.max())
print(f'CSR: {n:,} patents, {len(out_idx):,} edges | years {int(year.min())}..{YMAX}')

[453s] scanned 152,631,908 rows -> 119,298,022 utility<->utility edges
8,062,166 patents in citation graph; 119,105,013 unique directed edges (from 119,298,022 raw rows: 193,009 duplicates, 0 self-loops removed)
CSR: 8,062,166 patents, 119,105,013 edges | years 1976..2025


## 3. Numba engine — yearly n_i / n_j / n_k and F / E / G per focal patent, then prefix sums

In [4]:
from numba import njit, prange

@njit(inline='always')
def _bfind(arr, x):
    lo = 0; hi = len(arr)
    while lo < hi:
        mid = (lo + hi) >> 1
        if arr[mid] < x: lo = mid + 1
        else: hi = mid
    return lo < len(arr) and arr[lo] == x

@njit(parallel=True)
def trend_kernel(focal, off, out_ptr, out_idx, in_ptr, in_idx, year,
                 ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf):
    """Per focal document, per year since publication: new and cumulative n_i / n_j / n_k, CD,
    and the cumulative citer profile F / E / G.

    Dense layout: focal t owns slots off[t] .. off[t+1]-1, slot off[t]+d being year yF+d.
    ni/nj/nk receive the yearly increments, NI/NJ/NK the running totals, CD the running index,
    Ff/Ef/Gf the shares of the citers of age <= d that are Foundational / Extensional /
    Generalizational at age d. Slots of different focal documents never overlap, so the prange
    writes are race-free.

    Same definitions as the per-window engine (paper_disruption / patent_disruption):
      n_j(y) = citers published in year y that cite >= 1 reference of the focal;
      n_i(y) = the other citers of year y;
      n_k(y) = |B_y| - n_j(y), B_y = distinct documents of year y citing >= 1 reference of the
               focal, the focal itself excluded;
      citer c: up = |refs(F) & refs(c)|, down(d) = |{citers of F of age <= d} & refs(c)|;
               up > down -> E, down > up -> F, up = down = 0 -> G, up = down > 0 -> half F, half E.
    Summing over years 0..w reproduces ni_w / nj_w / nk_w / F_w / E_w / G_w exactly, because a
    window in the per-window engine is precisely a prefix of years. `down` grows with the age,
    which is why the profile is re-cut at every age instead of once.
    """
    for t in prange(len(focal)):
        F = focal[t]; yF = year[F]; b0 = off[t]; b1 = off[t + 1]; span = b1 - b0
        a0 = in_ptr[F]; a1 = in_ptr[F + 1]
        R = out_idx[out_ptr[F]:out_ptr[F + 1]]
        Rs = np.sort(R); As = np.sort(in_idx[a0:a1])
        # per-age citer-profile counters; a citer of age dc is present at every age >= dc
        Nc = np.zeros(span, np.int64); cF = np.zeros(span, np.int64); cE = np.zeros(span, np.int64)
        cG = np.zeros(span, np.int64); cT = np.zeros(span, np.int64)
        maxdeg = 0
        for ci in range(a0, a1):
            c = in_idx[ci]; deg = out_ptr[c + 1] - out_ptr[c]
            if deg > maxdeg: maxdeg = deg
        dds = np.empty(maxdeg, np.int64)
        for ci in range(a0, a1):
            c = in_idx[ci]; dc = year[c] - yF
            if dc < 0:
                continue
            up = 0; nd = 0
            for ri in range(out_ptr[c], out_ptr[c + 1]):
                d = out_idx[ri]
                if _bfind(Rs, d): up += 1
                if _bfind(As, d):
                    dd = year[d] - yF
                    if dd >= 0:
                        dds[nd] = dd; nd += 1
            if up > 0: nj[b0 + dc] += 1
            else:      ni[b0 + dc] += 1
            dsort = np.sort(dds[:nd]); p = 0
            for tt in range(dc, span):
                while p < nd and dsort[p] <= tt: p += 1     # down(tt) = co-cited citers of age <= tt
                Nc[tt] += 1
                if up > p:   cE[tt] += 1
                elif p > up: cF[tt] += 1
                elif up > 0: cT[tt] += 1
                else:        cG[tt] += 1
        # B: distinct citers of the focal's references (focal excluded), by year
        totB = 0
        for ri in range(len(R)):
            r = R[ri]; totB += in_ptr[r + 1] - in_ptr[r]
        if totB > 0:
            buf = np.empty(totB, np.int32); p = 0
            for ri in range(len(R)):
                r = R[ri]
                for j in range(in_ptr[r], in_ptr[r + 1]):
                    buf[p] = in_idx[j]; p += 1
            buf.sort(); prev = np.int32(-1)
            for ii in range(totB):
                b = buf[ii]
                if b == prev or b == F:
                    continue
                prev = b; dd = year[b] - yF
                if dd >= 0:
                    nk[b0 + dd] += 1
        # nk holds |B_y| so far; turn it into n_k(y) = |B_y| - n_j(y), then cumulate
        si = 0; sj = 0; sk = 0
        for q in range(span):
            p = b0 + q
            nk[p] -= nj[p]
            si += ni[p]; sj += nj[p]; sk += nk[p]
            NI[p] = si; NJ[p] = sj; NK[p] = sk
            den = si + sj + sk
            CD[p] = (si - sj) / den if den > 0 else np.nan
            if Nc[q] > 0:
                Ff[p] = (cF[q] + 0.5 * cT[q]) / Nc[q]
                Ef[p] = (cE[q] + 0.5 * cT[q]) / Nc[q]
                Gf[p] = cG[q] / Nc[q]
            else:
                Ff[p] = np.nan; Ef[p] = np.nan; Gf[p] = np.nan
print('engine ready')

engine ready


## 4. Run over all focal patents (with citers) in batches, stream to parquet, check every one against patent_disruption.parquet

In [5]:
%%time
focal_all = np.flatnonzero(in_ptr[1:] - in_ptr[:-1] > 0).astype(np.int64)
print(f'focal patents with >= 1 citer: {len(focal_all):,}  (batches of {BATCH:,})')
YMIN = int(year.min()); NY = YMAX - YMIN + 1
OUTC = ['CD', 'F', 'E', 'G']; CNTS = ['ni', 'nj', 'nk']
SCHEMA = pa.schema([('patent_id', pa.string()), ('grant_year', pa.int32()), ('cite_year', pa.int32()),
                    ('yrs_since_grant', pa.int32()),
                    ('ni_new', pa.int32()), ('nj_new', pa.int32()), ('nk_new', pa.int32()),
                    ('ni', pa.int32()), ('nj', pa.int32()), ('nk', pa.int32()),
                    ('CD', pa.float32()), ('F', pa.float32()), ('E', pa.float32()), ('G', pa.float32())])
# (grant_year, yrs_since_grant) accumulators for the summary in the next section; filled per batch so the
# summary never has to re-read the (large) output.
sum_o = {m: np.zeros(NY * NY, np.float64) for m in OUTC}; cnt_o = {m: np.zeros(NY * NY, np.int64) for m in OUTC}

# ── the per-window table, for the exhaustive check. Its rows are in code order (one row per
#    entry of the id array, written in that order), so row i is document i; the id column is
#    checked against the id array before anything is compared.
WIN = {'_3': 3, '_5': 5, '_10': 10, '_all': None}
_dt = pq.read_table(DISR_FP)
assert (np.asarray(_dt.column('patent_id').to_pandas()) == uni.astype(str)).all(), 'per-window table is not in code order'
DIS = {}
for s_ in WIN:
    for m in OUTC:
        DIS[m + s_] = _dt.column(m + s_).to_numpy(zero_copy_only=False).astype(np.float32)
    for m in CNTS:                     # -1 (paper) or NaN (patent) both mean "nothing in the window"
        v = _dt.column(m + s_).to_numpy(zero_copy_only=False).astype(np.float64)
        DIS[m + s_] = np.where(np.isfinite(v), v, -1).astype(np.int64)
del _dt; gc.collect()
print(f'per-window table loaded for the check: {len(DIS["CD_3"]):,} rows x {len(DIS)} columns')
mism = {s_: {m: 0 for m in OUTC + CNTS + ['undefined']} for s_ in WIN}
ncmp = {s_: 0 for s_ in WIN}

TMP = OUT_FP + '.tmp'
writer = pq.ParquetWriter(TMP, SCHEMA, compression='zstd')
rows_total = 0; docs_total = 0; t0 = time.time()
nb = (len(focal_all) + BATCH - 1) // BATCH
for bi, s in enumerate(range(0, len(focal_all), BATCH)):
    focal = focal_all[s:s + BATCH]
    yF = year[focal].astype(np.int64)
    span = YMAX - yF + 1
    off = np.zeros(len(focal) + 1, np.int64); np.cumsum(span, out=off[1:])
    T = int(off[-1])
    ni = np.zeros(T, np.int32); nj = np.zeros(T, np.int32); nk = np.zeros(T, np.int32)
    NI = np.empty(T, np.int32); NJ = np.empty(T, np.int32); NK = np.empty(T, np.int32)
    CD = np.empty(T, np.float32); Ff = np.empty(T, np.float32); Ef = np.empty(T, np.float32); Gf = np.empty(T, np.float32)
    trend_kernel(focal, off, out_ptr, out_idx, in_ptr, in_idx, year, ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf)

    # ── exhaustive check: the cumulative slot at age w IS the per-window value at w. When the
    #    document's span ends before w the last slot holds the total, which is also the window.
    OUT_ARR = {'CD': CD, 'F': Ff, 'E': Ef, 'G': Gf, 'ni': NI, 'nj': NJ, 'nk': NK}
    for s_, w in WIN.items():
        slot = off[:-1] + (span - 1 if w is None else np.minimum(w, span - 1))
        undefined = (NI[slot] == 0) & (NJ[slot] == 0) & (NK[slot] == 0)      # nothing in the window
        ref_undef = DIS['ni' + s_][focal] == -1
        mism[s_]['undefined'] += int((undefined != ref_undef).sum())
        ok = ~undefined & ~ref_undef
        for m in CNTS:
            mism[s_][m] += int((OUT_ARR[m][slot][ok] != DIS[m + s_][focal][ok]).sum())
        for m in OUTC:
            mism[s_][m] += int((~np.isclose(OUT_ARR[m][slot][ok], DIS[m + s_][focal][ok], atol=1e-6, equal_nan=True)).sum())
        ncmp[s_] += int(ok.sum())

    # one row per (patent, year) with any event: a new citer, or a new co-citing document
    ridx = np.flatnonzero((ni != 0) | (nj != 0) | (nk != 0))
    fpos = np.searchsorted(off, ridx, side='right') - 1          # focal position of each row
    yrs = (ridx - off[fpos]).astype(np.int32)
    pub = yF[fpos].astype(np.int32)
    ids = pa.array(uni[focal].astype(str))
    tbl = pa.table({'patent_id': ids.take(pa.array(fpos)), 'grant_year': pub, 'cite_year': pub + yrs, 'yrs_since_grant': yrs,
                    'ni_new': ni[ridx], 'nj_new': nj[ridx], 'nk_new': nk[ridx],
                    'ni': NI[ridx], 'nj': NJ[ridx], 'nk': NK[ridx],
                    'CD': CD[ridx], 'F': Ff[ridx], 'E': Ef[ridx], 'G': Gf[ridx]}, schema=SCHEMA)
    writer.write_table(tbl)
    # CD is NaN on the rare row whose denominator is 0 (a self-citing document), F/E/G are NaN
    # until the first citer arrives; the means are over defined values only, as in the by-year notebooks.
    key_all = (pub.astype(np.int64) - YMIN) * NY + yrs
    for m in OUTC:
        v = OUT_ARR[m][ridx]; fin = np.isfinite(v)
        sum_o[m] += np.bincount(key_all[fin], weights=v[fin].astype(np.float64), minlength=NY * NY)
        cnt_o[m] += np.bincount(key_all[fin], minlength=NY * NY)
    rows_total += len(ridx); docs_total += int((np.bincount(fpos, minlength=len(focal)) > 0).sum())
    print(f'[{time.time()-t0:6.0f}s] batch {bi+1:>3}/{nb}: {len(focal):,} focal -> '
          f'{len(ridx):,} rows  (cum {rows_total:,})', flush=True)
    del ni, nj, nk, NI, NJ, NK, CD, Ff, Ef, Gf, OUT_ARR, ridx, fpos, yrs, pub, ids, tbl, key_all, v, fin; gc.collect()
writer.close()
os.replace(TMP, OUT_FP)
del DIS; gc.collect()
print(f'WROTE {OUT_FP}  ({rows_total:,} rows, {os.path.getsize(OUT_FP)/1e9:.1f} GB)')
print(f'  patents covered: {docs_total:,} | cite_year {YMIN}..{YMAX}')

focal patents with >= 1 citer: 5,973,479  (batches of 1,000,000)
per-window table loaded for the check: 8,062,166 rows x 28 columns
[    80s] batch   1/6: 1,000,000 focal -> 10,343,275 rows  (cum 10,343,275)
[   110s] batch   2/6: 1,000,000 focal -> 25,751,304 rows  (cum 36,094,579)
[   137s] batch   3/6: 1,000,000 focal -> 22,381,159 rows  (cum 58,475,738)
[   169s] batch   4/6: 1,000,000 focal -> 17,073,393 rows  (cum 75,549,131)
[   210s] batch   5/6: 1,000,000 focal -> 11,487,207 rows  (cum 87,036,338)
[   254s] batch   6/6: 973,479 focal -> 6,679,973 rows  (cum 93,716,311)
WROTE /project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption_trend.parquet  (93,716,311 rows, 0.7 GB)
  patents covered: 5,973,424 | cite_year 1976..2025


In [6]:
# Every focal patent (not a sample), every window, every metric: the cumulative slot at age w
# against patent_disruption.parquet. `undefined` counts documents where one side says "nothing in the window"
# and the other does not.
print(f'{"window":<8}{"compared":>14}' + ''.join(f'{m:>10}' for m in OUTC + CNTS) + f'{"undefined":>12}')
print('-' * 92)
total_bad = 0
for s_ in WIN:
    print(f'{s_:<8}{ncmp[s_]:>14,}' + ''.join(f'{mism[s_][m]:>10,}' for m in OUTC + CNTS) + f'{mism[s_]["undefined"]:>12,}')
    total_bad += sum(mism[s_].values())
assert total_bad == 0, 'trend disagrees with the per-window table'
print(f'\n  cumulative trend == per-window CD / F / E / G / ni / nj / nk at every window, on all {len(focal_all):,} patents')

window        compared        CD         F         E         G        ni        nj        nk   undefined
--------------------------------------------------------------------------------------------
_3           5,606,555         0         0         0         0         0         0         0           0
_5           5,768,230         0         0         0         0         0         0         0           0
_10          5,899,905         0         0         0         0         0         0         0           0
_all         5,973,424         0         0         0         0         0         0         0           0

  cumulative trend == per-window CD / F / E / G / ni / nj / nk at every window, on all 5,973,479 patents


## 5. Summary — mean cumulative CD / F / E / G by age, pooled and by cohort; head of the output

In [7]:
cells = np.flatnonzero(sum(cnt_o.values()))
py, yy = np.divmod(cells, NY)
summ = pd.DataFrame({'grant_year': py + YMIN, 'yrs_since_grant': yy})
with np.errstate(invalid='ignore', divide='ignore'):
    for m in OUTC:
        summ[f'n_{m}'] = cnt_o[m][cells]
        summ[f'{m}_mean'] = sum_o[m][cells] / cnt_o[m][cells]
summ.to_parquet(SUM_FP, index=False)
print(f'WROTE {SUM_FP}  ({len(summ):,} (grant_year, yrs_since_grant) cells)')

# Mean cumulative CD / F / E / G by years since the patent year, all patents pooled. Reading down a
# column is how each outcome moves with the citation window.
S = {m: sum_o[m].reshape(NY, NY) for m in OUTC}; C = {m: cnt_o[m].reshape(NY, NY) for m in OUTC}
YRS = [0, 1, 2, 3, 5, 10, 15, 20, 30]
pooled = pd.DataFrame({'yrs_since_grant': YRS, 'n_CD': [int(C['CD'][:, y].sum()) for y in YRS], 'n_FEG': [int(C['F'][:, y].sum()) for y in YRS],
                       **{f'{m}_mean': [S[m][:, y].sum() / max(C[m][:, y].sum(), 1) for y in YRS] for m in OUTC}})
display(pooled.round(4))

# By cohort: each outcome at fixed ages, so cohorts are compared at the same window length.
rows = []
for lo, hi in ((1976, 1979), (1980, 1989), (1990, 1999), (2000, 2009), (2010, 2019)):
    if hi < YMIN or lo > YMAX:
        continue
    r = slice(max(lo, YMIN) - YMIN, hi - YMIN + 1)
    rows.append({'cohort': f'{lo}-{hi}', 'n_rows': int(C['CD'][r].sum()),
                 **{f'{m}@{y}': (S[m][r, y].sum() / C[m][r, y].sum() if C[m][r, y].sum() else np.nan)
                    for m in OUTC for y in (1, 3, 5, 10, 20)}})
COH = pd.DataFrame(rows).set_index('cohort')
for m in OUTC:
    print(f'\n{m} by cohort at fixed ages:')
    display(COH[[f'{m}@{y}' for y in (1, 3, 5, 10, 20)]].round(4))

head = pq.ParquetFile(OUT_FP).read_row_group(0).to_pandas()
with pd.option_context('display.width', 250, 'display.max_columns', 20):
    display(head.head(12))

WROTE /project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption_trend_summary.parquet  (1,275 (grant_year, yrs_since_grant) cells)


,yrs_since_grant,n_CD,n_FEG,CD_mean,F_mean,E_mean,G_mean
0,0,4792298,209412,0.0051,0.0009,0.4522,0.5469
1,1,4953390,1551360,0.0347,0.0047,0.4250,0.5703
2,2,4997870,2758243,0.0558,0.0141,0.4065,0.5794
3,3,4929278,3443906,0.0659,0.0273,0.3889,0.5838
4,5,4670305,3988920,0.0749,0.0568,0.3618,0.5814
5,10,3707845,3573936,0.0774,0.1284,0.3221,0.5496
6,15,2738991,2701816,0.0854,0.1894,0.2893,0.5213
7,20,1992262,1979807,0.1033,0.2432,0.2546,0.5021
8,30,748083,746423,0.1772,0.3212,0.1824,0.4964



CD by cohort at fixed ages:


,CD@1,CD@3,CD@5,CD@10,CD@20
cohort,,,,,
1976-1979,0.4478,0.6149,0.6472,0.6614,0.6747
1980-1989,0.0646,0.1450,0.1684,0.1838,0.1945
1990-1999,0.0226,0.0549,0.0681,0.0763,0.0786
2000-2009,0.0119,0.0238,0.0303,0.0361,0.0357
2010-2019,0.0274,0.0512,0.0611,0.0471,NaN



F by cohort at fixed ages:


...


E by cohort at fixed ages:


,E@1,E@3,E@5,E@10,E@20
cohort,,,,,
1976-1979,0.0706,0.0836,0.0782,0.0704,0.0583
1980-1989,0.3037,0.2652,0.2451,0.2165,0.1821
1990-1999,0.3838,0.3523,0.3298,0.2959,0.2520
2000-2009,0.4234,0.4127,0.3944,0.3567,0.3108
2010-2019,0.4823,0.4322,0.3961,0.3605,NaN



G by cohort at fixed ages:


,G@1,G@3,G@5,G@10,G@20
cohort,,,,,
1976-1979,0.9278,0.8934,0.8603,0.8015,0.7088
1980-1989,0.6941,0.7106,0.6977,0.6529,0.5803
1990-1999,0.6140,0.6231,0.6098,0.5599,0.4876
2000-2009,0.5736,0.5655,0.5567,0.5199,0.4619
2010-2019,0.5107,0.5356,0.5437,0.5195,NaN


,patent_id,grant_year,cite_year,yrs_since_grant,ni_new,nj_new,nk_new,ni,nj,nk,CD,F,E,G
0,10000000,2018,2018,0,0,0,6,0,0,6,0.000000,NaN,NaN,NaN
1,10000000,2018,2019,1,0,0,8,0,0,14,0.000000,NaN,NaN,NaN
2,10000000,2018,2020,2,2,1,8,2,1,22,0.040000,0.000000,0.333333,0.666667
3,10000000,2018,2021,3,1,0,3,3,1,25,0.068966,0.000000,0.250000,0.750000
4,10000000,2018,2022,4,3,1,7,6,2,32,0.100000,0.125000,0.250000,0.625000
5,10000000,2018,2023,5,1,1,2,7,3,34,0.090909,0.200000,0.300000,0.500000
6,10000000,2018,2024,6,3,0,2,10,3,36,0.142857,0.384615,0.230769,0.384615
7,10000000,2018,2025,7,6,0,2,16,3,38,0.228070,0.421053,0.157895,0.421053
8,10000003,2018,2018,0,0,0,4,0,0,4,0.000000,NaN,NaN,NaN
9,10000003,2018,2020,2,1,0,2,1,0,6,0.142857,0.000000,0.000000,1.000000
